##EXERCISE 1


Користејќи го моделот LLaMA-2 со квантизација со 4bits и техниката RAG,
генерирајте одговор за секое прашање од податочното множество CommonSenseQA.
Од базата на знаење GenericsKB изберете контекст од вкупно 5 документи за секое
прашање со семантичко пребарување и векторска репрезентација на документите
со моделот all-MiniLM-L6-v2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q datasets evaluate sacrebleu bert-score

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util

In [ ]:
from huggingface_hub import login
login()

In [ ]:
model_id = "meta-llama/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto'
)

In [ ]:
model

In [ ]:
embedding_models = {
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
    "distilroberta": "sentence-transformers/all-distilroberta-v1"
}

embedder = SentenceTransformer(embedding_models["minilm"])
embedder

In [ ]:
commonSenseQA = load_dataset("tau/commonsense_qa", split="validation[:200]")

In [ ]:
commonSenseQA

In [ ]:
# list first five samples
for i in range(5):
    ex = commonSenseQA[i]
    print("Question:", ex["question"])
    print("Question concept:", ex["question_concept"])
    print("Choices:", list(zip(ex["choices"]["label"], ex["choices"]["text"])))
    print("Answer:", ex["answerKey"])
    print("=" * 50)

In [ ]:
def load_genericskb(limit=None):
    kb = load_dataset("generics_kb", split="train")
    if limit:
        kb = kb.select(range(limit))
    documents = [row["generic_sentence"] for row in kb]
    return documents

In [ ]:
documents = load_genericskb(limit=None)

In [ ]:
def encode_texts(texts, model_name=embedding_models["minilm"]):
    embedder = SentenceTransformer(model_name)
    embeddings = embedder.encode(texts, convert_to_tensor=True)
    return model, embeddings

In [ ]:
model, doc_embeddings = encode_texts(documents)

In [ ]:
doc_embeddings[:5]

In [ ]:
def retrieve_top_k(question, documents, doc_embeddings, embedder, k=5):
    query_emb = embedder.encode([question], convert_to_tensor=True)
    cosine_scores = util.cos_sim(query_emb, doc_embeddings)[0]
    top_results = torch.topk(cosine_scores, k=k)
    top_docs = [documents[idx] for idx in top_results.indices]
    top_scores = [cosine_scores[idx].item() for idx in top_results.indices]

    return top_docs, top_scores

In [ ]:
def format_prompt(example, context=None):
    choices_text = "\n".join(
        f"{label}. {text}" for label, text in zip(example["choices"]["label"], example["choices"]["text"])
    )
    prompt = f"Question: {example['question']}\nChoices:\n{choices_text}\nAnswer (one letter):"
    if context:
        prompt = f"Context: {context}\n\n{prompt}"
    return prompt

In [ ]:
def generate_answer(tokenizer, model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(text)
    if "Answer" in text:
        answer_part = text.split("Answer")[-1]
    else:
        answer_part = text
    for c in answer_part:
        if c in ["A", "B", "C", "D", "E"]:
            return c

    return "(not found)"

In [ ]:
def print_results(example, predicted, top_docs, top_scores=None):
    print("\n" + "="*80)
    print("QUESTION:", example["question"])
    print("Choices:", list(zip(example["choices"]["label"], example["choices"]["text"])))
    print("Gold Answer:", example["answerKey"])
    print("Predicted Answer:", predicted)
    print("\nTop documents retrieved as context:")
    for i, doc in enumerate(top_docs, 1):
        score_str = f" (score={top_scores[i-1]:.4f})" if top_scores is not None else ""
        print(f"{i}. {doc}{score_str}")
    print("=" * 90)

In [ ]:
from nltk.translate.bleu_score import sentence_bleu
from bert_score import score as bert_score
import numpy as np

In [ ]:
def evaluate(dataset, documents, doc_embeddings, embedder, tokenizer, model,
             k=5, use_rag=True, print_examples=False):
    refs, preds = [], []

    for ex in dataset:
        top_docs, top_scores = retrieve_top_k(ex["question"], documents, doc_embeddings, embedder, k) if use_rag else []
        context = " ".join(top_docs) if top_docs else None
        prompt = format_prompt(ex, context)
        pred = generate_answer(tokenizer, model, prompt)
        refs.append(ex["answerKey"])
        preds.append(pred)

        if print_examples:
            print_results(ex, pred, top_docs, top_scores)

    return refs, preds

In [ ]:
print("Evaluating with RAG (5 context docs)...")
refs, pred = evaluate(commonSenseQA, documents, doc_embeddings, embedder, tokenizer, model,
                             k=5, use_rag=True, print_examples=True)

In [ ]:
import evaluate as ev
bleu = ev.load("bleu")
bert = ev.load("bertscore")

In [ ]:
def letters_to_texts(preds, refs, dataset):
    pred_texts = []
    ref_texts = []
    for p, r, ex in zip(preds, refs, dataset):
        mapping = {label: text for label, text in zip(ex["choices"]["label"], ex["choices"]["text"])}
        pred_texts.append(mapping[p])
        ref_texts.append(mapping[r])
    return pred_texts, ref_texts

pred_texts, ref_texts = letters_to_texts(pred, refs, commonSenseQA)
bleu_score = bleu.compute(predictions=pred_texts, references=[[r] for r in ref_texts])
bert_score = np.mean(bert.compute(predictions=pred_texts, references=ref_texts, lang="en")['f1'])
print("BLEU (text):", bleu_score)
print("BERTScore F1 (text):", bert_score)

The evaluation results indicate that the RAG approach with LLaMA-2 (4-bit quantized) performs very well on the CommonSenseQA task. The BLEU score of 47.6% shows that the generated answers have substantial n-gram overlap with the reference answers, which is particularly impressive given that many answers are short phrases or even single words. The breakdown of n-gram precisions demonstrates that the model consistently captures key words and partial sequences, while minor drops for 3-grams and 4-grams are expected due to the brevity of the answers. More importantly, the BERTScore F1 of 0.947 reflects a very high semantic similarity between the predicted and reference answers, indicating that the model reliably captures the intended meaning even when exact word matches are not perfect. Overall, these results suggest that combining RAG with LLaMA-2 and a semantic retrieval mechanism allows for accurate and semantically coherent question answering, making it a robust approach for short-answer QA tasks like CommonSenseQA.

Многу време и ресурси(GPU)(usage limits) е да го пробам и со другиот embedding model, меѓутоа значително подобри резултати се добиваат со дополнителниот контекст од документи кои се даваат на моделот за збогатување на знаењето.

## EXERCISE 2

Користејќи го моделот LLaMA-2 со квантизација со 4bits и техниката RAG,
генерирајте одговор за секое прашање од податочното множество RAG-MiniWikipedia. Изберете контекст од вкупно 5 документи за секое прашање со
семантичко пребарување и векторска репрезентација на документите со моделот
all-MiniLM-L6-v2.

In [ ]:
wiki = load_dataset(
    "rag-datasets/rag-mini-wikipedia",
    "question-answer",
    split="test[:200]"
)

In [ ]:
wiki.column_names

In [ ]:
# list first five samples
for i in range(5):
    ex = wiki[i]
    print("Question:", ex["question"])
    print("Answer:", ex["answer"])

In [ ]:
def format_prompt(example, context=None):
    prompt = f"Question: {example['question']}\nAnswer:"
    if context:
        prompt = f"Context: {context}\n\n{prompt}"
    print(prompt)
    return prompt

def generate_answer(tokenizer, model, prompt, max_new_tokens=50):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    if "Answer" in text:
        answer = text.split("Answer")[-1].strip()
    else:
        answer = text.strip()
    return answer

In [ ]:
def print_results(example, predicted, top_docs, top_scores):
    print("\n" + "="*80)
    print("QUESTION:", example["question"])
    print("Gold Answer:", example["answer"])
    print("Predicted Answer:", predicted)
    print("Top documents retrieved as context:")
    for i, (doc, score) in enumerate(zip(top_docs, top_scores), 1):
        print(f"{i}. ({score:.3f}) {doc}")
    print("\n" + "="*150)

In [ ]:
def evaluate(dataset, documents, doc_embeddings, embedder, tokenizer, model,
             k=5, use_rag=True, print_examples=False):
    refs, preds = [], []

    for ex in dataset:
        top_docs, top_scores = retrieve_top_k(ex["question"], documents, doc_embeddings, embedder, k) if use_rag else []
        context = " ".join(top_docs) if top_docs else None
        prompt = format_prompt(ex, context)
        pred = generate_answer(tokenizer, model, prompt)
        refs.append(ex["answer"])
        preds.append(pred)

        if print_examples:
            print_results(ex, pred, top_docs, top_scores)

    return refs, preds

The model occasionally generates contradictory discourse markers (e.g., starting an answer with “No” while providing a factually correct explanation). This behavior stems from the autoregressive nature of large language models, which optimize for token-level likelihood rather than global logical consistency. To ensure reliable evaluation, answers were constrained and normalized during post-processing.

In [ ]:
print("Evaluating with RAG (5 context docs)...")
refs, pred = evaluate(wiki, documents[:], doc_embeddings[:], embedder, tokenizer, model,
                             k=5, use_rag=True, print_examples=True)

In [ ]:
bleu_score = bleu.compute(predictions=pred, references=[[r] for r in refs])
bert_score = np.mean(bert.compute(predictions=pred, references=refs, lang="en")['f1'])
print("BLEU (text):", bleu_score)
print("BERTScore F1 (text):", bert_score)

The results for the Wikipedia RAG task indicate that the model is not optimized for precise short-answer generation without downstream fine-tuning. The extremely low BLEU (0.6%) reflects that the predicted answers have almost no word overlap with references, likely due to verbose outputs. However, the BERTScore F1 of 0.82 shows that the model still captures the general semantic content of the answers. Overall, this suggests that LLaMA-2 with 4-bit quantization can produce semantically relevant but overly long or imprecise answers in a zero-shot setting. To improve results, the model would benefit from fine-tuning on RAG-style QA with Wikipedia passages or better prompt engineering to generate concise, precise answers.

##EXERCISE 3

Користејќи го моделот LLaMA-2 со квантизација со 4bits и техниката RAG,
трансформирајте ги речениците кои содржат негативен сентимент од податочното
множество Yelp_parallel во реченици со позитивен сентимент. Изберете контекст
од вкупно 5 документи за секое прашање со семантичко пребарување и векторска
репрезентација на документите со моделот all-MiniLM-L6-v2.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/NLP2025/yelp_parallel/yelp_parallel/test_en_parallel.txt', sep='\t', header=None)
df.columns = ["Negative", "Positive"]
df = df.iloc[1:].reset_index(drop=True)
df.head(6)

In [ ]:
yelp = load_dataset(
    "csv",
    data_files="/content/drive/MyDrive/NLP2025/yelp_parallel/yelp_parallel/test_en_parallel.txt",
    delimiter="\t",
    split="train"
)
yelp

In [ ]:
print(yelp[0]["Style 1"])
print(yelp[0]["Style 2"])

In [ ]:
def format_prompt(example, context=None):
    prompt = ""
    if context:
        prompt += "Context (positive examples):\n"
        for i, doc in enumerate(context, 1):
            prompt += f"{i}. {doc}\n"
        prompt += "\n"
    prompt += f"Negative review: {example['Style 1']}\n"
    prompt += "Rewrite it to be POSITIVE in **one sentence**:\n"

    return prompt

def generate_answer(tokenizer, model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=60, do_sample=False)
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    if "Rewrite it to be POSITIVE in **one sentence**:" in text:
        return text.split("Rewrite it to be POSITIVE in **one sentence**:")[-1].strip()
    return text.strip()

def print_results(example, predicted, top_docs, top_scores):
    print("\n" + "="*80)
    print("Negative:", example["Style 1"])
    print("Gold Positive:", example["Style 2"])
    print("Predicted Positive:", predicted)
    print("\nTop documents retrieved as context:")
    for i, (doc, score) in enumerate(zip(top_docs, top_scores), 1):
        print(f"{i}. ({score:.3f}) {doc}")

In [ ]:
def evaluate_dataset(dataset, documents, doc_embeddings, embedder, tokenizer, model, k=5, use_rag=True, print_examples=True):
    refs, preds = [], []

    for ex in dataset:
        top_docs, top_scores = retrieve_top_k(ex["Style 1"], documents, doc_embeddings, embedder, k) if use_rag else ([], [])
        context = top_docs if top_docs else None
        prompt = format_prompt(ex, context)
        pred = generate_answer(tokenizer, model, prompt)
        refs.append(ex["Style 2"])
        preds.append(pred)

        if print_examples:
            print_results(ex, pred, top_docs, top_scores)

    return refs, preds

In [ ]:
print("Evaluating with RAG (5 context docs)...")
refs, pred = evaluate_dataset(yelp, documents[:], doc_embeddings[:], embedder, tokenizer, model,
                             k=5, use_rag=True, print_examples=True)

In [ ]:
bleu_score = bleu.compute(predictions=pred, references=[[r] for r in refs])
bert_score = np.mean(bert.compute(predictions=pred, references=refs, lang="en")['f1'])
print("BLEU (text):", bleu_score)
print("BERTScore F1 (text):", bert_score)

The results show that the RAG-LLaMA-2 model effectively transforms negative Yelp reviews into positive sentences. The BLEU score is low (0.04), reflecting minimal exact word overlap, mainly due to longer, more elaborate model outputs. However, the BERTScore F1 of 0.88 indicates strong semantic alignment with the references, demonstrating that the model successfully captures the intended positive sentiment. Overall, while BLEU underestimates performance, the approach reliably produces semantically correct positive rewrites, with room for improvement in brevity and style consistency.